# AI Stock：Google Colab GPU 訓練

本 notebook 直接呼叫專案現有的 PyTorch training service。請先把整個 `AI_stock` 資料夾上傳到 Google Drive 的 `MyDrive/AI_stock`，再依序執行下列儲存格。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/AI_stock'
import os
if not os.path.isdir(PROJECT_DIR):
    raise FileNotFoundError(f'找不到 {PROJECT_DIR}，請先將專案資料夾上傳到 Google Drive。')
os.chdir(PROJECT_DIR)
print('Project:', os.getcwd())

In [ ]:
# 保留 Colab 預裝的 CUDA 版 PyTorch，只安裝專案的其他依賴。
import subprocess, sys
with open('requirements.txt', encoding='utf-8') as file:
    packages = [line.strip() for line in file if line.strip() and not line.startswith('#') and line.strip().lower() != 'torch']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencies installed:', len(packages))

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('目前不是 GPU runtime。請到 Runtime > Change runtime type > T4 GPU 後重新執行。')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

In [ ]:
# 訓練設定：base=主模型，stock=個股模型，gnn=市場 ST-GNN。
MODE = 'stock'
TICKERS = ['2330.TW', '0050.TW']
EPOCHS = 100
BASIC_BASE_MODEL = False
print(MODE, TICKERS, EPOCHS)

In [ ]:
import subprocess, sys
command = [sys.executable, 'scripts/colab_train.py', '--mode', MODE]
if MODE in ('stock', 'gnn'):
    command += ['--tickers', *TICKERS]
if MODE == 'gnn':
    command += ['--epochs', str(EPOCHS)]
if MODE == 'base' and BASIC_BASE_MODEL:
    command.append('--basic')
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

訓練結果會寫回 Drive：個股模型在 `data/saved_models/`，主模型是 `data/saved_models/base_model.pth`，訓練記錄在 `data/training_records/`。若要換模式，只修改設定儲存格後重新執行最後一格。